In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict, deque
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, CatBoostClassifier

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

## 1. Path dan konfigurasi


In [2]:
BASE_PATH = Path.home() / "Downloads" / "Gammafest"
DATA_PATH = BASE_PATH / "dataset"
OUTPUT_DIR = BASE_PATH / "experiments" / "percobaan 9 - dixon coles pseudo"

TRAIN_PATH = DATA_PATH / "train.csv"
TEST_PATH = DATA_PATH / "test.csv"
SAMPLE_PATH = DATA_PATH / "sample submission.csv"

SUBMISSION_PATH = OUTPUT_DIR / "submission_dixon_coles_pseudo_seq.csv"
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / "submission_dixon_coles_pseudo_roundclip.csv"
VALID_REPORT_PATH = OUTPUT_DIR / "validation_dixon_coles_regression_report.csv"
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / "postprocess_dixon_coles_report.csv"

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = "GPU"
MAX_SCORE = 6
N_RANDOM_BLENDS = 2000

ELO_INIT = 1500.0
ELO_K_DEFAULT = 20
ELO_K_IMPORTANT = 40
IMPORTANT_TOURNAMENTS = {
    "FIFA World Cup", "AFC Asian Cup", "AFC Championship", "UEFA Euro",
    "Africa Cup of Nations", "African Cup of Nations", "Copa America", "Copa América",
    "Gold Cup", "CONCACAF Gold Cup"
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 9 - dixon coles pseudo


## 2. Load data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

train_raw["date_dt"] = pd.to_datetime(train_raw["date"], errors="coerce")
test_raw["date_dt"] = pd.to_datetime(test_raw["date"], errors="coerce")

print("train:", train_raw.shape, train_raw["date_dt"].min(), "->", train_raw["date_dt"].max())
print("test :", test_raw.shape, test_raw["date_dt"].min(), "->", test_raw["date_dt"].max())
print("sample:", sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 48) 1872-11-30 00:00:00 -> 2011-08-04 00:00:00
test : (42422, 21) 2011-08-06 00:00:00 -> 2026-03-31 00:00:00
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169,2011-08-06
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169,2011-08-06


## 3. Static features, temporal split, dan AW-MAE constants

In [4]:
CAT_COLS = [
    "gender", "team", "opponent", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]

STATIC_NUM_COLS = [
    "is_home", "neutral",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp", "temperature_venue",
]

DATE_FEATURES = ["year", "month", "dayofweek"]

TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Championship": 1.80,
    "AFC Asian Cup": 1.80,
    "UEFA Euro": 1.80,
    "Copa America": 1.80,
    "Copa América": 1.80,
    "Africa Cup of Nations": 1.80,
    "African Cup of Nations": 1.80,
    "Gold Cup": 1.75,
    "CONCACAF Gold Cup": 1.75,
    "FIFA World Cup qualification": 1.50,
    "UEFA Euro qualification": 1.40,
    "AFC Asian Cup qualification": 1.40,
    "Friendly": 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def add_basic_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["date"], errors="coerce")
    df["date_dt"] = dt
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["tournament_weight"] = df["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT)
    return df


match_dates = train_raw.groupby("match_id")["date_dt"].min().sort_values()
split_idx = int(len(match_dates) * (1 - VALID_FRAC))
train_match_ids = set(match_dates.index[:split_idx])
val_match_ids = set(match_dates.index[split_idx:])

tr_raw = train_raw[train_raw["match_id"].isin(train_match_ids)].copy()
val_raw = train_raw[train_raw["match_id"].isin(val_match_ids)].copy()

print("Train fold rows:", tr_raw.shape, tr_raw["date_dt"].min(), "->", tr_raw["date_dt"].max())
print("Valid fold rows:", val_raw.shape, val_raw["date_dt"].min(), "->", val_raw["date_dt"].max())
print("Train matches:", len(train_match_ids), "Valid matches:", len(val_match_ids))

Train fold rows: (63016, 48) 1872-11-30 00:00:00 -> 2005-01-30 00:00:00
Valid fold rows: (15756, 48) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00
Train matches: 31508 Valid matches: 7878


## 4. Forward-only reconstruction

`can_update_outcome=True` hanya untuk history train yang boleh memperbarui state. Validation/test tidak memperbarui form, H2H, atau Elo dengan target mereka.
Schedule tetap memperbarui last-match date karena tanggal pertandingan memang diketahui.

In [5]:
def points_from_score(gf, ga):
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0


def elo_k(tournament):
    return ELO_K_IMPORTANT if tournament in IMPORTANT_TOURNAMENTS else ELO_K_DEFAULT


def summarize_form(history):
    if len(history) == 0:
        return {
            "points_last5": np.nan,
            "points_last10": np.nan,
            "gd_last5": np.nan,
            "avg_goals_last5": np.nan,
            "avg_conceded_last5": np.nan,
            "win_rate_last10": np.nan,
            "matches_known": 0,
        }
    last5 = list(history)[-5:]
    last10 = list(history)[-10:]
    return {
        "points_last5": float(sum(x["points"] for x in last5)),
        "points_last10": float(sum(x["points"] for x in last10)),
        "gd_last5": float(sum(x["gd"] for x in last5)),
        "avg_goals_last5": float(np.mean([x["gf"] for x in last5])),
        "avg_conceded_last5": float(np.mean([x["ga"] for x in last5])),
        "win_rate_last10": float(np.mean([x["points"] == 3 for x in last10])),
        "matches_known": int(len(history)),
    }


def build_reconstructed_features(input_df):
    df = add_basic_features(input_df).copy()
    df["_orig_order"] = np.arange(len(df))
    if "can_update_outcome" not in df.columns:
        df["can_update_outcome"] = df["team_goals"].notna() if "team_goals" in df.columns else False
    if "split_name" not in df.columns:
        df["split_name"] = "unknown"

    for col in ["team_goals", "opp_goals", "rank_team", "rank_opponent"]:
        if col not in df.columns:
            df[col] = np.nan

    df = df.sort_values(["date_dt", "match_id", "Id"]).reset_index(drop=True)

    form = defaultdict(lambda: deque(maxlen=50))
    h2h = defaultdict(lambda: deque(maxlen=20))
    elo = defaultdict(lambda: ELO_INIT)
    last_match_date = {}
    latest_rank = {}
    feature_rows = []

    for match_id, grp in df.groupby("match_id", sort=False):
        match_date = grp["date_dt"].iloc[0]

        for _, row in grp.iterrows():
            gender = row["gender"]
            team = row["team"]
            opp = row["opponent"]
            team_key = (gender, team)
            opp_key = (gender, opp)
            pair_key = (gender, team, opp)

            tf = summarize_form(form[team_key])
            of = summarize_form(form[opp_key])
            hhist = list(h2h[pair_key])[-5:]

            h2h_points = float(sum(x["points"] for x in hhist)) if hhist else np.nan
            h2h_gd = float(sum(x["gd"] for x in hhist)) if hhist else np.nan
            h2h_matches = int(len(hhist))

            team_last_date = last_match_date.get(team_key)
            opp_last_date = last_match_date.get(opp_key)
            days_team = (match_date - team_last_date).days if team_last_date is not None and pd.notna(match_date) else np.nan
            days_opp = (match_date - opp_last_date).days if opp_last_date is not None and pd.notna(match_date) else np.nan

            rank_team_latest = latest_rank.get(team_key, np.nan)
            rank_opp_latest = latest_rank.get(opp_key, np.nan)

            feature_rows.append({
                "_orig_order": row["_orig_order"],
                "team_points_last5_recon": tf["points_last5"],
                "opp_points_last5_recon": of["points_last5"],
                "points_last5_diff_recon": tf["points_last5"] - of["points_last5"] if pd.notna(tf["points_last5"]) and pd.notna(of["points_last5"]) else np.nan,
                "team_points_last10_recon": tf["points_last10"],
                "opp_points_last10_recon": of["points_last10"],
                "points_last10_diff_recon": tf["points_last10"] - of["points_last10"] if pd.notna(tf["points_last10"]) and pd.notna(of["points_last10"]) else np.nan,
                "team_gd_last5_recon": tf["gd_last5"],
                "opp_gd_last5_recon": of["gd_last5"],
                "gd_last5_diff_recon": tf["gd_last5"] - of["gd_last5"] if pd.notna(tf["gd_last5"]) and pd.notna(of["gd_last5"]) else np.nan,
                "team_avg_goals_last5_recon": tf["avg_goals_last5"],
                "team_avg_conceded_last5_recon": tf["avg_conceded_last5"],
                "opp_avg_goals_last5_recon": of["avg_goals_last5"],
                "opp_avg_conceded_last5_recon": of["avg_conceded_last5"],
                "avg_goals_diff_recon": tf["avg_goals_last5"] - of["avg_goals_last5"] if pd.notna(tf["avg_goals_last5"]) and pd.notna(of["avg_goals_last5"]) else np.nan,
                "avg_conceded_diff_recon": tf["avg_conceded_last5"] - of["avg_conceded_last5"] if pd.notna(tf["avg_conceded_last5"]) and pd.notna(of["avg_conceded_last5"]) else np.nan,
                "team_win_rate_last10_recon": tf["win_rate_last10"],
                "opp_win_rate_last10_recon": of["win_rate_last10"],
                "win_rate_diff_recon": tf["win_rate_last10"] - of["win_rate_last10"] if pd.notna(tf["win_rate_last10"]) and pd.notna(of["win_rate_last10"]) else np.nan,
                "h2h_points_last5_recon": h2h_points,
                "h2h_gd_last5_recon": h2h_gd,
                "h2h_matches_last5_recon": h2h_matches,
                "days_since_last_match_team_recon": days_team,
                "days_since_last_match_opp_recon": days_opp,
                "days_since_last_match_diff_recon": days_team - days_opp if pd.notna(days_team) and pd.notna(days_opp) else np.nan,
                "elo_team_recon": float(elo[team_key]),
                "elo_opponent_recon": float(elo[opp_key]),
                "elo_diff_recon": float(elo[team_key] - elo[opp_key]),
                "rank_team_latest_recon": rank_team_latest,
                "rank_opponent_latest_recon": rank_opp_latest,
                "rank_diff_latest_recon": rank_team_latest - rank_opp_latest if pd.notna(rank_team_latest) and pd.notna(rank_opp_latest) else np.nan,
                "rank_missing_team_recon": int(pd.isna(rank_team_latest)),
                "rank_missing_opp_recon": int(pd.isna(rank_opp_latest)),
                "team_matches_known_recon": tf["matches_known"],
                "opp_matches_known_recon": of["matches_known"],
                "matches_known_diff_recon": tf["matches_known"] - of["matches_known"],
            })

        for _, row in grp.iterrows():
            last_match_date[(row["gender"], row["team"])] = match_date

        can_update = bool(grp["can_update_outcome"].fillna(False).all())
        has_scores = grp["team_goals"].notna().all() and grp["opp_goals"].notna().all()
        if can_update and has_scores:
            for _, row in grp.iterrows():
                gender = row["gender"]
                team = row["team"]
                opp = row["opponent"]
                gf = float(row["team_goals"])
                ga = float(row["opp_goals"])
                pts = points_from_score(gf, ga)
                gd = gf - ga
                team_key = (gender, team)
                form[team_key].append({"points": pts, "gf": gf, "ga": ga, "gd": gd})
                h2h[(gender, team, opp)].append({"points": pts, "gd": gd})

                if pd.notna(row.get("rank_team", np.nan)):
                    latest_rank[team_key] = float(row["rank_team"])
                if pd.notna(row.get("rank_opponent", np.nan)):
                    latest_rank[(gender, opp)] = float(row["rank_opponent"])

            row = grp.iloc[0]
            gender = row["gender"]
            team_key = (gender, row["team"])
            opp_key = (gender, row["opponent"])
            e_team = elo[team_key]
            e_opp = elo[opp_key]
            expected_team = 1 / (1 + 10 ** ((e_opp - e_team) / 400))
            gf = float(row["team_goals"])
            ga = float(row["opp_goals"])
            actual_team = 1.0 if gf > ga else 0.5 if gf == ga else 0.0
            k = elo_k(row["tournament"])
            elo[team_key] = e_team + k * (actual_team - expected_team)
            elo[opp_key] = e_opp + k * ((1 - actual_team) - (1 - expected_team))

    feature_df = pd.DataFrame(feature_rows)
    return df.merge(feature_df, on="_orig_order", how="left").sort_values("_orig_order").reset_index(drop=True)

## 5. Build features untuk eval dan final test

In [6]:
tr_for_eval = tr_raw.copy()
tr_for_eval["split_name"] = "train_fold"
tr_for_eval["can_update_outcome"] = True

val_for_eval = val_raw.copy()
val_for_eval["split_name"] = "valid_fold"
val_for_eval["can_update_outcome"] = False

eval_input = pd.concat([tr_for_eval, val_for_eval], ignore_index=True, sort=False)
print("Building eval reconstructed features...")
eval_recon = build_reconstructed_features(eval_input)
tr_recon = eval_recon[eval_recon["split_name"] == "train_fold"].copy()
val_recon = eval_recon[eval_recon["split_name"] == "valid_fold"].copy()

full_train_for_final = train_raw.copy()
full_train_for_final["split_name"] = "full_train"
full_train_for_final["can_update_outcome"] = True

test_for_final = test_raw.copy()
test_for_final["split_name"] = "test"
test_for_final["can_update_outcome"] = False
test_for_final["team_goals"] = np.nan
test_for_final["opp_goals"] = np.nan

final_input = pd.concat([full_train_for_final, test_for_final], ignore_index=True, sort=False)
print("Building final reconstructed features...")
final_recon = build_reconstructed_features(final_input)
full_train_recon = final_recon[final_recon["split_name"] == "full_train"].copy()
test_recon = final_recon[final_recon["split_name"] == "test"].copy()

print("tr_recon:", tr_recon.shape)
print("val_recon:", val_recon.shape)
print("full_train_recon:", full_train_recon.shape)
print("test_recon:", test_recon.shape)

Building eval reconstructed features...
Building final reconstructed features...
tr_recon: (63016, 90)
val_recon: (15756, 90)
full_train_recon: (78772, 90)
test_recon: (42422, 90)


## 6. Sanity check dan preprocessing matrix

In [7]:
RECON_FEATURES = [
    "team_points_last5_recon", "opp_points_last5_recon", "points_last5_diff_recon",
    "team_points_last10_recon", "opp_points_last10_recon", "points_last10_diff_recon",
    "team_gd_last5_recon", "opp_gd_last5_recon", "gd_last5_diff_recon",
    "team_avg_goals_last5_recon", "team_avg_conceded_last5_recon",
    "opp_avg_goals_last5_recon", "opp_avg_conceded_last5_recon",
    "avg_goals_diff_recon", "avg_conceded_diff_recon",
    "team_win_rate_last10_recon", "opp_win_rate_last10_recon", "win_rate_diff_recon",
    "h2h_points_last5_recon", "h2h_gd_last5_recon", "h2h_matches_last5_recon",
    "days_since_last_match_team_recon", "days_since_last_match_opp_recon", "days_since_last_match_diff_recon",
    "elo_team_recon", "elo_opponent_recon", "elo_diff_recon",
    "rank_team_latest_recon", "rank_opponent_latest_recon", "rank_diff_latest_recon",
    "rank_missing_team_recon", "rank_missing_opp_recon",
    "team_matches_known_recon", "opp_matches_known_recon", "matches_known_diff_recon",
]

compare_pairs = [
    ("team_points_last5", "team_points_last5_recon"),
    ("opp_points_last5", "opp_points_last5_recon"),
    ("team_points_last10", "team_points_last10_recon"),
    ("opp_points_last10", "opp_points_last10_recon"),
    ("team_gd_last5", "team_gd_last5_recon"),
    ("opp_gd_last5", "opp_gd_last5_recon"),
    ("team_avg_goals_last5", "team_avg_goals_last5_recon"),
    ("team_avg_conceded_last5", "team_avg_conceded_last5_recon"),
    ("opp_avg_goals_last5", "opp_avg_goals_last5_recon"),
    ("opp_avg_conceded_last5", "opp_avg_conceded_last5_recon"),
    ("team_win_rate_last10", "team_win_rate_last10_recon"),
    ("opp_win_rate_last10", "opp_win_rate_last10_recon"),
    ("h2h_points_last5", "h2h_points_last5_recon"),
    ("h2h_gd_last5", "h2h_gd_last5_recon"),
    ("elo_team", "elo_team_recon"),
    ("elo_opponent", "elo_opponent_recon"),
]

rows = []
for orig, recon in compare_pairs:
    if orig in full_train_recon.columns and recon in full_train_recon.columns:
        tmp = full_train_recon[[orig, recon]].dropna()
        corr = tmp[orig].corr(tmp[recon]) if len(tmp) > 2 else np.nan
        rows.append({"original": orig, "reconstructed": recon, "non_null_pairs": len(tmp), "corr": corr})
compare_df = pd.DataFrame(rows).sort_values("corr")
display(compare_df)

BASE_NUM_COLS = STATIC_NUM_COLS + DATE_FEATURES + ["tournament_weight"] + RECON_FEATURES


def finalize_feature_frames(train_df, other_df, fit_name="train"):
    train = train_df.copy()
    other = other_df.copy()

    for df in [train, other]:
        for col in CAT_COLS:
            df[col] = df[col].fillna("Unknown").astype(str)
        for col in BASE_NUM_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in BASE_NUM_COLS:
        flag_col = f"{col}_missing"
        train[flag_col] = train[col].isna().astype(int)
        other[flag_col] = other[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[BASE_NUM_COLS].median(numeric_only=True)
    for col in BASE_NUM_COLS:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        other[col] = other[col].fillna(fill_value)

    feature_cols = CAT_COLS + BASE_NUM_COLS + missing_flag_cols
    cat_feature_indices = [feature_cols.index(c) for c in CAT_COLS]
    print(f"{fit_name}: features={len(feature_cols)}, missing train={train[feature_cols].isna().sum().sum()}, missing other={other[feature_cols].isna().sum().sum()}")
    return train, other, feature_cols, cat_feature_indices


tr_model, val_model, FEATURE_COLS, CAT_FEATURE_INDICES = finalize_feature_frames(tr_recon, val_recon, fit_name="eval")
full_train_model, test_model, FINAL_FEATURE_COLS, FINAL_CAT_FEATURE_INDICES = finalize_feature_frames(full_train_recon, test_recon, fit_name="final")

assert FEATURE_COLS == FINAL_FEATURE_COLS
assert CAT_FEATURE_INDICES == FINAL_CAT_FEATURE_INDICES

X_tr = tr_model[FEATURE_COLS]
X_val = val_model[FEATURE_COLS]
y_tr_team = tr_model["team_goals"]
y_tr_opp = tr_model["opp_goals"]
y_val_team = val_model["team_goals"]
y_val_opp = val_model["opp_goals"]

X_full = full_train_model[FEATURE_COLS]
y_full_team = full_train_model["team_goals"]
y_full_opp = full_train_model["opp_goals"]
X_test = test_model[FEATURE_COLS]

print("X_tr:", X_tr.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

,original,reconstructed,non_null_pairs,corr
8,opp_avg_goals_last5,opp_avg_goals_last5_recon,78289,0.800315
5,opp_gd_last5,opp_gd_last5_recon,78289,0.811221
9,opp_avg_conceded_last5,opp_avg_conceded_last5_recon,78289,0.824315
4,team_gd_last5,team_gd_last5_recon,78289,0.839995
1,opp_points_last5,opp_points_last5_recon,78289,0.841340
6,team_avg_goals_last5,team_avg_goals_last5_recon,78289,0.842844
7,team_avg_conceded_last5,team_avg_conceded_last5_recon,78289,0.843728
11,opp_win_rate_last10,opp_win_rate_last10_recon,78289,0.868597
3,opp_points_last10,opp_points_last10_recon,78289,0.875067
0,team_points_last5,team_points_last5_recon,78289,0.890021


eval: features=105, missing train=0, missing other=0
final: features=105, missing train=0, missing other=0
X_tr: (63016, 105) X_val: (15756, 105) X_test: (42422, 105)


## 7. AW-MAE dan post-processing helpers

In [8]:
def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    return np.clip(team, 0, max_score), np.clip(opp, 0, max_score)


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true["team_goals"].to_numpy()
    true_opp = df_true["opp_goals"].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()
    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        "AW-MAE": score,
        "MAE_raw_component": mae.mean(),
        "exact_acc": exact.mean(),
        "outcome_acc": outcome.mean(),
        "goal_diff_acc": gd.mean(),
    }
    return score, diag


def evaluate_raw_predictions(name, df_true, team_raw, opp_raw, max_score=MAX_SCORE):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    return {"model": name, **diag}

## 8. Train CatBoost ensemble

In [9]:
CATBOOST_CONFIGS = [
    {
        "name": "recon_cat_mae_d7_seed42",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1600, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "recon_cat_mae_d6_seed7",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1800, learning_rate=0.035, depth=6, l2_leaf_reg=9, random_strength=1.6, bagging_temperature=0.8, random_seed=7),
    },
    {
        "name": "recon_cat_mae_d8_seed99",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1400, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.3, random_seed=99),
    },
    {
        "name": "recon_cat_rmse_d7_seed123",
        "params": dict(loss_function="RMSE", eval_metric="MAE", iterations=1500, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

val_pred_bank = {}
trained_val_models = {}
reports = []

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=160)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=160)

    pred_t = np.clip(mt.predict(X_val), 0, None)
    pred_o = np.clip(mo.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (mt, mo)

    report = evaluate_raw_predictions(name, val_model, pred_t, pred_o, max_score=MAX_SCORE)
    report["best_iter_team"] = mt.best_iteration_
    report["best_iter_opp"] = mo.best_iteration_
    reports.append(report)
    print(report)

print("Training minutes:", (time.time() - start) / 60)
reports_df = pd.DataFrame(reports).sort_values("AW-MAE")
reports_df.to_csv(VALID_REPORT_PATH, index=False)
display(reports_df)
print("Saved:", VALID_REPORT_PATH)


Training recon_cat_mae_d7_seed42


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747216	test: 1.1324730	best: 1.1324730 (0)	total: 30.5ms	remaining: 48.7s
150:	learn: 1.0764290	test: 1.0519558	best: 1.0519558 (150)	total: 2.83s	remaining: 27.1s
300:	learn: 1.0436786	test: 1.0319599	best: 1.0319599 (300)	total: 5.63s	remaining: 24.3s
450:	learn: 1.0284858	test: 1.0255289	best: 1.0255250 (449)	total: 8.47s	remaining: 21.6s
600:	learn: 1.0189926	test: 1.0229603	best: 1.0229603 (600)	total: 11.3s	remaining: 18.8s
750:	learn: 1.0123772	test: 1.0220525	best: 1.0220380 (748)	total: 14.1s	remaining: 15.9s
900:	learn: 1.0065711	test: 1.0218579	best: 1.0218436 (899)	total: 17s	remaining: 13.2s
1050:	learn: 1.0011882	test: 1.0208404	best: 1.0208238 (1048)	total: 19.8s	remaining: 10.4s
1200:	learn: 0.9961422	test: 1.0203000	best: 1.0202895 (1192)	total: 22.7s	remaining: 7.55s
1350:	learn: 0.9915984	test: 1.0198084	best: 1.0197886 (1339)	total: 25.7s	remaining: 4.74s
1500:	learn: 0.9870842	test: 1.0194336	best: 1.0193943 (1489)	total: 28.6s	remaining: 1.89s
1599:	l

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747479	test: 1.1325134	best: 1.1325134 (0)	total: 17.5ms	remaining: 28s
150:	learn: 1.0770789	test: 1.0521239	best: 1.0521239 (150)	total: 2.79s	remaining: 26.8s
300:	learn: 1.0444437	test: 1.0329329	best: 1.0329329 (300)	total: 5.59s	remaining: 24.1s
450:	learn: 1.0287081	test: 1.0255316	best: 1.0255316 (450)	total: 8.36s	remaining: 21.3s
600:	learn: 1.0197235	test: 1.0229127	best: 1.0229127 (600)	total: 11.1s	remaining: 18.5s
750:	learn: 1.0130423	test: 1.0217431	best: 1.0217195 (743)	total: 13.9s	remaining: 15.7s
900:	learn: 1.0067127	test: 1.0211802	best: 1.0211743 (899)	total: 16.7s	remaining: 13s
1050:	learn: 1.0015278	test: 1.0207128	best: 1.0207128 (1050)	total: 19.5s	remaining: 10.2s
1200:	learn: 0.9967527	test: 1.0202036	best: 1.0201964 (1199)	total: 22.3s	remaining: 7.42s
1350:	learn: 0.9921411	test: 1.0199147	best: 1.0199135 (1349)	total: 25.1s	remaining: 4.62s
1500:	learn: 0.9877069	test: 1.0197602	best: 1.0197254 (1493)	total: 27.9s	remaining: 1.84s
1599:	lea

...

0:	learn: 1.2750192	test: 1.2666925	best: 1.2666925 (0)	total: 19.2ms	remaining: 28.8s
150:	learn: 1.0449129	test: 1.0741839	best: 1.0740737 (148)	total: 3.13s	remaining: 28s
300:	learn: 1.0261658	test: 1.0734326	best: 1.0730031 (254)	total: 6s	remaining: 23.9s
450:	learn: 1.0123414	test: 1.0719646	best: 1.0716323 (423)	total: 8.9s	remaining: 20.7s
600:	learn: 1.0008799	test: 1.0696228	best: 1.0696228 (600)	total: 11.8s	remaining: 17.7s
750:	learn: 0.9909610	test: 1.0691558	best: 1.0689261 (733)	total: 14.8s	remaining: 14.7s
900:	learn: 0.9814738	test: 1.0693375	best: 1.0689196 (870)	total: 17.7s	remaining: 11.8s
1050:	learn: 0.9727849	test: 1.0695044	best: 1.0688689 (949)	total: 20.7s	remaining: 8.84s
bestTest = 1.068868854
bestIteration = 949
Shrink model to first 950 iterations.


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2745149	test: 1.2660927	best: 1.2660927 (0)	total: 16.7ms	remaining: 25s
150:	learn: 1.0449266	test: 1.0739298	best: 1.0735371 (111)	total: 2.82s	remaining: 25.2s
300:	learn: 1.0258933	test: 1.0733244	best: 1.0729356 (294)	total: 5.7s	remaining: 22.7s
450:	learn: 1.0123459	test: 1.0715696	best: 1.0714818 (448)	total: 8.61s	remaining: 20s
600:	learn: 0.9999715	test: 1.0709472	best: 1.0704420 (569)	total: 11.5s	remaining: 17.2s
bestTest = 1.070442039
bestIteration = 569
Shrink model to first 570 iterations.
{'model': 'recon_cat_rmse_d7_seed123', 'AW-MAE': np.float64(3.0877350745369223), 'MAE_raw_component': np.float64(1.0375412541254125), 'exact_acc': np.float64(0.09723280020309723), 'outcome_acc': np.float64(0.5692434628078192), 'goal_diff_acc': np.float64(0.23083269865448083), 'best_iter_team': 949, 'best_iter_opp': 569}
Training minutes: 3.5716508905092876


,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,recon_cat_rmse_d7_seed123,3.087735,1.037541,0.097233,0.569243,0.230833,949,569
0,recon_cat_mae_d7_seed42,3.109232,0.996922,0.110878,0.512249,0.234577,1585,1599
1,recon_cat_mae_d6_seed7,3.129564,1.000190,0.110117,0.510282,0.233689,1758,1783
2,recon_cat_mae_d8_seed99,3.130898,0.999461,0.110117,0.506029,0.235149,1393,971


Saved: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 9 - dixon coles pseudo\validation_dixon_coles_regression_report.csv


## 9. Blend search

In [10]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t_int, pred_o_int)
    return score, diag


blend_rows = []
best = {"score": np.inf, "weights": None, "name": None, "diag": None}

for i, name in enumerate(model_names):
    w = np.zeros(len(model_names)); w[i] = 1
    score, diag = score_blend(w)
    blend_rows.append({"blend": f"single_{name}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"single_{name}", "diag": diag}

w = np.ones(len(model_names)) / len(model_names)
score, diag = score_blend(w)
blend_rows.append({"blend": "equal_average", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "equal_average", "diag": diag}

single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag = score_blend(w)
blend_rows.append({"blend": "inverse_awmae", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "inverse_awmae", "diag": diag}

rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag = score_blend(w)
    if k < 25 or score < best["score"]:
        blend_rows.append({"blend": f"random_{k}_a{alpha}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"random_{k}_a{alpha}", "diag": diag}

blend_df = pd.DataFrame(blend_rows).sort_values("AW-MAE").reset_index(drop=True)
display(blend_df.head(20))

print("Best blend:", best["name"])
print("Best AW-MAE:", best["score"])
print("Diagnostics:", best["diag"])
print("Weights:")
for name, weight in sorted(zip(model_names, best["weights"]), key=lambda x: -x[1]):
    print(f"  {name:<32} {weight:.5f}")

,blend,AW-MAE,w_recon_cat_mae_d7_seed42,w_recon_cat_mae_d6_seed7,w_recon_cat_mae_d8_seed99,w_recon_cat_rmse_d7_seed123
0,random_1624_a0.25,3.064317,0.286636,0.000096,0.096515,0.616753
1,random_216_a0.25,3.064570,0.383465,0.001930,0.001622,0.612983
2,random_48_a0.25,3.066052,0.001429,0.178225,0.123840,0.696505
3,random_43_a2.0,3.068515,0.058658,0.204089,0.102791,0.634461
4,random_31_a2.0,3.070105,0.110881,0.151535,0.142076,0.595508
5,random_5_a0.5,3.070476,0.026982,0.100146,0.299397,0.573474
6,random_9_a0.5,3.078497,0.197650,0.000592,0.028711,0.773046
7,single_recon_cat_rmse_d7_seed123,3.087735,0.000000,0.000000,0.000000,1.000000
8,random_22_a1.0,3.095916,0.398764,0.116891,0.179967,0.304377
9,random_1_a0.5,3.096048,0.017968,0.150514,0.490905,0.340613


Best blend: random_1624_a0.25
Best AW-MAE: 3.064317217919381
Diagnostics: {'AW-MAE': np.float64(3.064317217919381), 'MAE_raw_component': np.float64(1.0122810357958874), 'exact_acc': np.float64(0.103008377760853), 'outcome_acc': np.float64(0.5463950241177964), 'goal_diff_acc': np.float64(0.23502157908098503)}
Weights:
  recon_cat_rmse_d7_seed123        0.61675
  recon_cat_mae_d7_seed42          0.28664
  recon_cat_mae_d8_seed99          0.09652
  recon_cat_mae_d6_seed7           0.00010


## 10. Outcome classifier win/draw/loss

Regressor menebak jumlah gol, tapi AW-MAE sangat menghukum outcome yang salah.

Cell ini menambahkan beberapa `CatBoostClassifier` untuk memprediksi outcome dari perspektif row:
- `0`: team kalah,
- `1`: seri,
- `2`: team menang.

Probabilitas classifier akan dipakai di post-processing skor integer.

In [11]:
def make_outcome_target(df):
    diff = df["team_goals"].to_numpy() - df["opp_goals"].to_numpy()
    return np.where(diff > 0, 2, np.where(diff < 0, 0, 1)).astype(int)


def aligned_outcome_proba(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), 3), dtype=float)
    for j, cls in enumerate(model.classes_):
        out[:, int(cls)] = raw[:, j]
    row_sum = out.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return out / row_sum


y_tr_outcome = make_outcome_target(tr_model)
y_val_outcome = make_outcome_target(val_model)
y_full_outcome = make_outcome_target(full_train_model)

CLASSIFIER_CONFIGS = [
    {
        "name": "outcome_cls_d6_seed42",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1200, learning_rate=0.045, depth=6, l2_leaf_reg=6, random_strength=1.0, random_seed=42),
    },
    {
        "name": "outcome_cls_d7_seed7",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1000, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.4, random_seed=7),
    },
    {
        "name": "outcome_cls_d5_seed99",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1400, learning_rate=0.035, depth=5, l2_leaf_reg=5, random_strength=1.8, random_seed=99),
    },
]

BASE_CLS_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

outcome_val_proba_bank = {}
trained_val_classifiers = {}
cls_reports = []

start = time.time()
for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    clf = CatBoostClassifier(**params)
    clf.fit(
        X_tr,
        y_tr_outcome,
        cat_features=CAT_FEATURE_INDICES,
        eval_set=(X_val, y_val_outcome),
        use_best_model=True,
        early_stopping_rounds=140,
    )

    proba = aligned_outcome_proba(clf, X_val)
    pred = proba.argmax(axis=1)
    acc = (pred == y_val_outcome).mean()
    nll = -np.log(np.clip(proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean()

    outcome_val_proba_bank[name] = proba
    trained_val_classifiers[name] = clf
    cls_reports.append({"classifier": name, "outcome_acc": acc, "nll": nll, "best_iter": clf.best_iteration_})
    print(cls_reports[-1])

cls_report_df = pd.DataFrame(cls_reports).sort_values(["nll", "outcome_acc"], ascending=[True, False])
display(cls_report_df)
print("Classifier training minutes:", (time.time() - start) / 60)

outcome_model_names = list(outcome_val_proba_bank.keys())
outcome_val_proba = np.mean([outcome_val_proba_bank[name] for name in outcome_model_names], axis=0)
outcome_val_pred = outcome_val_proba.argmax(axis=1)
print("Avg classifier outcome_acc:", (outcome_val_pred == y_val_outcome).mean())
print("Avg classifier nll:", -np.log(np.clip(outcome_val_proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean())


Training outcome_cls_d6_seed42
0:	learn: 1.0830117	test: 1.0820565	best: 1.0820565 (0)	total: 14.7ms	remaining: 17.6s
150:	learn: 0.8786135	test: 0.8826689	best: 0.8826689 (150)	total: 1.13s	remaining: 7.82s
300:	learn: 0.8626315	test: 0.8788702	best: 0.8788702 (300)	total: 2.19s	remaining: 6.55s
450:	learn: 0.8523470	test: 0.8785181	best: 0.8784005 (399)	total: 3.29s	remaining: 5.46s
bestTest = 0.8784004903
bestIteration = 399
Shrink model to first 400 iterations.
{'classifier': 'outcome_cls_d6_seed42', 'outcome_acc': np.float64(0.5990733688753491), 'nll': np.float64(0.8784004804123147), 'best_iter': 399}

Training outcome_cls_d7_seed7
0:	learn: 1.0846232	test: 1.0838907	best: 1.0838907 (0)	total: 9.09ms	remaining: 9.08s
150:	learn: 0.8749491	test: 0.8810945	best: 0.8810945 (150)	total: 1.42s	remaining: 7.99s
300:	learn: 0.8579520	test: 0.8777172	best: 0.8777172 (300)	total: 2.81s	remaining: 6.52s
450:	learn: 0.8453869	test: 0.8769001	best: 0.8767865 (438)	total: 4.2s	remaining: 5.11

,classifier,outcome_acc,nll,best_iter
1,outcome_cls_d7_seed7,0.599073,0.876786,438
2,outcome_cls_d5_seed99,0.598312,0.878186,586
0,outcome_cls_d6_seed42,0.599073,0.878400,399


Classifier training minutes: 0.2505139112472534
Avg classifier outcome_acc: 0.599009900990099
Avg classifier nll: 0.8770994458515005


## 11. CatBoost Poisson lambda models

In [12]:
POISSON_CONFIGS = [
    {
        "name": "pois_d6_seed42",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1400, learning_rate=0.040, depth=6, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "pois_d7_seed7",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1200, learning_rate=0.035, depth=7, l2_leaf_reg=9, random_strength=1.3, bagging_temperature=0.7, random_seed=7),
    },
    {
        "name": "pois_d5_seed99",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1600, learning_rate=0.030, depth=5, l2_leaf_reg=6, random_strength=1.8, bagging_temperature=0.9, random_seed=99),
    },
]

BASE_POIS_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)


def predict_poisson_lambda(model, X):
    # CatBoost Poisson uses an exponential link. Explicit Exponent prediction keeps lambda positive.
    try:
        pred = model.predict(X, prediction_type="Exponent")
    except TypeError:
        pred = model.predict(X)
        if np.nanmin(pred) <= 0:
            pred = np.exp(np.clip(pred, -7, 3))
    pred = np.asarray(pred, dtype=float)
    return np.clip(pred, 0.03, 8.0)


poisson_val_bank = {}
trained_val_poisson = {}
poisson_reports = []

start = time.time()
for cfg in POISSON_CONFIGS:
    name = cfg["name"]
    params = {**BASE_POIS_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training Poisson", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=150)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=150)

    lam_t = predict_poisson_lambda(mt, X_val)
    lam_o = predict_poisson_lambda(mo, X_val)
    poisson_val_bank[name] = (lam_t, lam_o)
    trained_val_poisson[name] = (mt, mo)

    pred_t, pred_o = postprocess_round_clip(lam_t, lam_o, max_score=MAX_SCORE)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    poisson_reports.append({"model": name, **diag, "best_iter_team": mt.best_iteration_, "best_iter_opp": mo.best_iteration_})
    print(poisson_reports[-1])

poisson_report_df = pd.DataFrame(poisson_reports).sort_values("AW-MAE")
display(poisson_report_df)
print("Poisson training minutes:", (time.time() - start) / 60)

poisson_model_names = list(poisson_val_bank.keys())
poisson_team_matrix = np.vstack([poisson_val_bank[name][0] for name in poisson_model_names])
poisson_opp_matrix = np.vstack([poisson_val_bank[name][1] for name in poisson_model_names])
poisson_val_team_lambda = np.mean(poisson_team_matrix, axis=0)
poisson_val_opp_lambda = np.mean(poisson_opp_matrix, axis=0)


Training Poisson pois_d6_seed42
0:	learn: 0.9765347	test: 0.9801893	best: 0.9801893 (0)	total: 17.6ms	remaining: 24.6s
150:	learn: 0.5819989	test: 0.6649641	best: 0.6649641 (150)	total: 2.96s	remaining: 24.5s
300:	learn: 0.5603131	test: 0.6564221	best: 0.6564221 (300)	total: 6.1s	remaining: 22.3s
450:	learn: 0.5467686	test: 0.6538331	best: 0.6537611 (434)	total: 9.2s	remaining: 19.4s
600:	learn: 0.5351661	test: 0.6521343	best: 0.6521090 (599)	total: 12.3s	remaining: 16.3s
750:	learn: 0.5256166	test: 0.6504863	best: 0.6504863 (750)	total: 15.5s	remaining: 13.4s
900:	learn: 0.5174395	test: 0.6496426	best: 0.6496326 (897)	total: 18.9s	remaining: 10.4s
1050:	learn: 0.5094316	test: 0.6483470	best: 0.6482140 (1024)	total: 21.9s	remaining: 7.27s
1200:	learn: 0.5024262	test: 0.6481021	best: 0.6480199 (1185)	total: 25.4s	remaining: 4.21s
1350:	learn: 0.4952959	test: 0.6476306	best: 0.6476306 (1350)	total: 28.7s	remaining: 1.04s
1399:	learn: 0.4932862	test: 0.6474484	best: 0.6473687 (1394)	tota

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
1,pois_d7_seed7,3.015624,1.017771,0.099073,0.572798,0.236418,1187,1157
0,pois_d6_seed42,3.036374,1.023927,0.096789,0.570767,0.234514,1394,944
2,pois_d5_seed99,3.058933,1.030433,0.094758,0.569942,0.232356,1222,1584


Poisson training minutes: 2.759797736008962


## 12. Bivariate Poisson + Dixon-Coles expected AW-MAE selector

In [13]:
def poisson_factorials(max_score):
    facts = np.ones(max_score + 1, dtype=float)
    for i in range(1, max_score + 1):
        facts[i] = facts[i - 1] * i
    return facts


def bivariate_poisson_distribution(lambda_team, lambda_opp, shared_lambda=0.0, max_score=6):
    lambda_team = np.clip(np.asarray(lambda_team, dtype=float), 0.03, 8.0)
    lambda_opp = np.clip(np.asarray(lambda_opp, dtype=float), 0.03, 8.0)
    lam3 = np.minimum(float(shared_lambda), 0.80 * np.minimum(lambda_team, lambda_opp))
    lam1 = np.clip(lambda_team - lam3, 1e-8, None)
    lam2 = np.clip(lambda_opp - lam3, 1e-8, None)

    states = np.array([(i, j) for i in range(max_score + 1) for j in range(max_score + 1)], dtype=int)
    facts = poisson_factorials(max_score)
    probs = np.zeros((len(lambda_team), len(states)), dtype=float)
    exp_term = np.exp(-(lam1 + lam2 + lam3))

    for s, (x, y) in enumerate(states):
        total = np.zeros(len(lambda_team), dtype=float)
        for k in range(min(x, y) + 1):
            total += (
                (lam1 ** (x - k)) / facts[x - k]
                * (lam2 ** (y - k)) / facts[y - k]
                * (lam3 ** k) / facts[k]
            )
        probs[:, s] = exp_term * total

    probs = np.clip(probs, 1e-300, None)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs, states


def apply_dixon_coles_correction(probs, states, lambda_team, lambda_opp, rho=0.0):
    if abs(float(rho)) < 1e-12:
        return probs

    probs = probs.copy()
    lambda_team = np.asarray(lambda_team, dtype=float)
    lambda_opp = np.asarray(lambda_opp, dtype=float)
    rho = float(rho)

    for idx, (x, y) in enumerate(states):
        if x == 0 and y == 0:
            tau = 1 - lambda_team * lambda_opp * rho
        elif x == 0 and y == 1:
            tau = 1 + lambda_team * rho
        elif x == 1 and y == 0:
            tau = 1 + lambda_opp * rho
        elif x == 1 and y == 1:
            tau = np.full(len(lambda_team), 1 - rho, dtype=float)
        else:
            continue
        probs[:, idx] *= np.clip(tau, 1e-6, 10.0)

    probs = np.clip(probs, 1e-300, None)
    return probs / probs.sum(axis=1, keepdims=True)


def state_outcome_class(states):
    diff = states[:, 0] - states[:, 1]
    return np.where(diff > 0, 2, np.where(diff < 0, 0, 1))


def build_score_pair_prior(df, max_score=6, smoothing=1.0):
    counts = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)
    team = np.clip(df["team_goals"].round().astype(int).to_numpy(), 0, max_score)
    opp = np.clip(df["opp_goals"].round().astype(int).to_numpy(), 0, max_score)
    for tg, og in zip(team, opp):
        counts[tg, og] += 1.0
    return (counts / counts.sum()).reshape(-1)


def awmae_loss_matrix(states):
    cand_team = states[:, 0][:, None]
    cand_opp = states[:, 1][:, None]
    true_team = states[:, 0][None, :]
    true_opp = states[:, 1][None, :]

    mae = (np.abs(true_team - cand_team) + np.abs(true_opp - cand_opp)) / 2
    exact = ((true_team == cand_team) & (true_opp == cand_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(cand_team, cand_opp)).astype(int)
    gd = ((true_team - true_opp) == (cand_team - cand_opp)).astype(int)
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    return ((mae + penalty) * multiplier) ** 1.5


def calibrate_score_distribution(probs, states, outcome_proba=None, classifier_power=0.0, score_prior=None, prior_power=0.0):
    probs = probs.copy()
    if outcome_proba is not None and classifier_power > 0:
        cls = state_outcome_class(states)
        probs *= np.clip(outcome_proba[:, cls], 1e-12, 1.0) ** classifier_power
    if score_prior is not None and prior_power > 0:
        probs *= np.clip(score_prior, 1e-12, 1.0)[None, :] ** prior_power
    probs = np.clip(probs, 1e-300, None)
    return probs / probs.sum(axis=1, keepdims=True)


def select_by_expected_loss(probs, states):
    loss = awmae_loss_matrix(states)
    expected = probs @ loss.T
    best_idx = np.argmin(expected, axis=1)
    return states[best_idx, 0].astype(int), states[best_idx, 1].astype(int)


def dixon_bivariate_select(team_raw, opp_raw, pois_team, pois_opp, outcome_proba, params, train_prior_df):
    max_score = int(params["max_score"])
    lambda_blend = float(params["lambda_blend"])
    lam_team = np.clip((1 - lambda_blend) * team_raw + lambda_blend * pois_team, 0.03, 8.0)
    lam_opp = np.clip((1 - lambda_blend) * opp_raw + lambda_blend * pois_opp, 0.03, 8.0)

    probs, states = bivariate_poisson_distribution(lam_team, lam_opp, shared_lambda=params["shared_lambda"], max_score=max_score)
    probs = apply_dixon_coles_correction(probs, states, lam_team, lam_opp, rho=params.get("dixon_rho", 0.0))
    score_prior = build_score_pair_prior(train_prior_df, max_score=max_score, smoothing=1.0)
    probs = calibrate_score_distribution(
        probs,
        states,
        outcome_proba=outcome_proba,
        classifier_power=params["classifier_power"],
        score_prior=score_prior,
        prior_power=params["prior_power"],
    )
    return select_by_expected_loss(probs, states)


blend_weights = best["weights"] / best["weights"].sum()
reg_val_team_raw = np.average(team_matrix, axis=0, weights=blend_weights)
reg_val_opp_raw = np.average(opp_matrix, axis=0, weights=blend_weights)

base_clip_rows = []
for max_score in range(4, 9):
    pred_t, pred_o = postprocess_round_clip(reg_val_team_raw, reg_val_opp_raw, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    base_clip_rows.append({"max_score": max_score, **diag})
base_clip_df = pd.DataFrame(base_clip_rows).sort_values("AW-MAE")
display(base_clip_df)

base_max_score = int(base_clip_df.iloc[0]["max_score"])
base_pred_t, base_pred_o = postprocess_round_clip(reg_val_team_raw, reg_val_opp_raw, max_score=base_max_score)
base_score, base_diag = compute_awmae(val_model, base_pred_t, base_pred_o)
print("Round+clip best:", base_score, "max_score:", base_max_score)

max_score_candidates = sorted(set([base_max_score, max(4, base_max_score - 1), min(8, base_max_score + 1)]))
lambda_blends = [0.00, 0.25, 0.50, 0.75, 1.00]
shared_lambdas = [0.00, 0.03, 0.06, 0.10, 0.15]
classifier_powers = [0.00, 0.50, 1.00, 1.75]
prior_powers = [0.00, 0.25, 0.50]
dixon_rhos = [-0.15, -0.10, -0.05, 0.00, 0.05, 0.10, 0.15]

pp_rows = []
best_pp = {
    "score": base_score,
    "params": {
        "mode": "round_clip",
        "max_score": base_max_score,
        "lambda_blend": 0.0,
        "shared_lambda": 0.0,
        "classifier_power": 0.0,
        "prior_power": 0.0,
        "dixon_rho": 0.0,
    },
    "diag": base_diag,
}

start = time.time()
for max_score in max_score_candidates:
    for lambda_blend in lambda_blends:
        lam_team = np.clip((1 - lambda_blend) * reg_val_team_raw + lambda_blend * poisson_val_team_lambda, 0.03, 8.0)
        lam_opp = np.clip((1 - lambda_blend) * reg_val_opp_raw + lambda_blend * poisson_val_opp_lambda, 0.03, 8.0)
        for shared_lambda in shared_lambdas:
            raw_probs, states = bivariate_poisson_distribution(lam_team, lam_opp, shared_lambda=shared_lambda, max_score=max_score)
            for dixon_rho in dixon_rhos:
                dc_probs = apply_dixon_coles_correction(raw_probs, states, lam_team, lam_opp, rho=dixon_rho)
                score_prior = build_score_pair_prior(tr_model, max_score=max_score, smoothing=1.0)
                for classifier_power in classifier_powers:
                    for prior_power in prior_powers:
                        probs = calibrate_score_distribution(
                            dc_probs,
                            states,
                            outcome_proba=outcome_val_proba,
                            classifier_power=classifier_power,
                            score_prior=score_prior,
                            prior_power=prior_power,
                        )
                        pred_t, pred_o = select_by_expected_loss(probs, states)
                        score, diag = compute_awmae(val_model, pred_t, pred_o)
                        params = {
                            "mode": "dixon_bivar_expected_awmae",
                            "max_score": max_score,
                            "lambda_blend": lambda_blend,
                            "shared_lambda": shared_lambda,
                            "classifier_power": classifier_power,
                            "prior_power": prior_power,
                            "dixon_rho": dixon_rho,
                        }
                        pp_rows.append({**params, **diag})
                        if score < best_pp["score"]:
                            best_pp = {"score": score, "params": params.copy(), "diag": diag.copy()}

pp_df = pd.DataFrame(pp_rows).sort_values("AW-MAE").reset_index(drop=True)
pp_df.to_csv(POSTPROCESS_REPORT_PATH, index=False)

BEST_PP_PARAMS = best_pp["params"]
BEST_PP_SCORE = best_pp["score"]
BEST_MAX_SCORE = int(BEST_PP_PARAMS["max_score"])

print("Dixon-Coles search minutes:", (time.time() - start) / 60)
print("Best score:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Best diagnostics:", best_pp["diag"])
display(pp_df.head(25))

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,4,3.054041,1.009838,0.104405,0.546395,0.236989
1,5,3.060165,1.011202,0.102247,0.546395,0.234260
2,6,3.064317,1.012281,0.103008,0.546395,0.235022
3,7,3.069843,1.014217,0.101929,0.546395,0.234070
4,8,3.072890,1.015327,0.101993,0.546395,0.234133


Round+clip best: 3.0540406451772024 max_score: 4
Dixon-Coles search minutes: 0.8828255931536356
Best score: 2.9177677928160892
Best params: {'mode': 'dixon_bivar_expected_awmae', 'max_score': 4, 'lambda_blend': 1.0, 'shared_lambda': 0.15, 'classifier_power': 0.0, 'prior_power': 0.0, 'dixon_rho': 0.0}
Best diagnostics: {'AW-MAE': np.float64(2.9177677928160892), 'MAE_raw_component': np.float64(1.0079652195988829), 'exact_acc': np.float64(0.11348057882711349), 'outcome_acc': np.float64(0.592028433612592), 'goal_diff_acc': np.float64(0.2387027164254887)}


,mode,max_score,lambda_blend,shared_lambda,classifier_power,prior_power,dixon_rho,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,dixon_bivar_expected_awmae,4,1.00,0.15,0.0,0.00,0.00,2.917768,1.007965,0.113481,0.592028,0.238703
1,dixon_bivar_expected_awmae,5,1.00,0.15,0.0,0.00,-0.05,2.917964,1.004633,0.113417,0.593044,0.236989
2,dixon_bivar_expected_awmae,4,1.00,0.06,0.0,0.00,-0.05,2.918959,1.009012,0.114179,0.591457,0.239655
3,dixon_bivar_expected_awmae,4,1.00,0.15,0.0,0.00,0.05,2.919351,1.008156,0.113163,0.592473,0.236101
4,dixon_bivar_expected_awmae,5,1.00,0.10,0.0,0.25,0.15,2.919369,1.009362,0.113734,0.591711,0.235656
5,dixon_bivar_expected_awmae,5,1.00,0.03,0.0,0.00,-0.10,2.919529,1.006632,0.112655,0.592854,0.237179
6,dixon_bivar_expected_awmae,5,1.00,0.06,0.0,0.00,-0.10,2.919670,1.005585,0.113100,0.592219,0.237116
7,dixon_bivar_expected_awmae,4,1.00,0.15,0.0,0.00,0.10,2.919723,1.009076,0.112275,0.593298,0.234323
8,dixon_bivar_expected_awmae,5,1.00,0.03,0.0,0.25,0.15,2.919828,1.011266,0.113608,0.592917,0.236354
9,dixon_bivar_expected_awmae,4,1.00,0.15,0.0,0.00,-0.10,2.919876,1.004506,0.114623,0.586316,0.239718


## 13. Train final models, stage1 Dixon-Coles, dan pseudo-sequential stage2

In [14]:
STAGE1_PATH = OUTPUT_DIR / "submission_dixon_coles_stage1.csv"
PSEUDO_PATH = OUTPUT_DIR / "submission_dixon_coles_pseudo_seq.csv"

final_pred_bank = {}
final_models = {}

for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 350)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final regression training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_pred_bank[name] = (np.clip(mt.predict(X_test), 0, None), np.clip(mo.predict(X_test), 0, None))
    final_models[name] = (mt, mo)

final_outcome_probas = []
final_classifiers = {}
for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    val_clf = trained_val_classifiers[name]
    best_iter = int(getattr(val_clf, "best_iteration_", 700) + 120)
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 250)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final classifier training", name, "iterations:", params["iterations"])
    clf = CatBoostClassifier(**params)
    clf.fit(X_full, y_full_outcome, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_outcome_probas.append(aligned_outcome_proba(clf, X_test))
    final_classifiers[name] = clf

test_outcome_proba = np.mean(final_outcome_probas, axis=0)

final_poisson_bank = {}
final_poisson_models = {}
for cfg in POISSON_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_poisson[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_POIS_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 300)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final Poisson training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_poisson_bank[name] = (predict_poisson_lambda(mt, X_test), predict_poisson_lambda(mo, X_test))
    final_poisson_models[name] = (mt, mo)


def average_regression_predictions(models_bank, X):
    team_preds = []
    opp_preds = []
    for name in model_names:
        mt, mo = models_bank[name]
        team_preds.append(np.clip(mt.predict(X), 0, None))
        opp_preds.append(np.clip(mo.predict(X), 0, None))
    team_matrix_local = np.vstack(team_preds)
    opp_matrix_local = np.vstack(opp_preds)
    weights = best["weights"] / best["weights"].sum()
    return np.average(team_matrix_local, axis=0, weights=weights), np.average(opp_matrix_local, axis=0, weights=weights)


def average_poisson_predictions(models_bank, X):
    team_preds = []
    opp_preds = []
    for name in poisson_model_names:
        mt, mo = models_bank[name]
        team_preds.append(predict_poisson_lambda(mt, X))
        opp_preds.append(predict_poisson_lambda(mo, X))
    return np.mean(np.vstack(team_preds), axis=0), np.mean(np.vstack(opp_preds), axis=0)


def average_classifier_predictions(models_bank, X):
    probas = []
    for name in outcome_model_names:
        probas.append(aligned_outcome_proba(models_bank[name], X))
    return np.mean(probas, axis=0)


reg_test_team_raw, reg_test_opp_raw = average_regression_predictions(final_models, X_test)
poisson_test_team_lambda, poisson_test_opp_lambda = average_poisson_predictions(final_poisson_models, X_test)

round_team, round_opp = postprocess_round_clip(reg_test_team_raw, reg_test_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[["Id"]].copy()
submission_round["team_goals"] = round_team
submission_round["opp_goals"] = round_opp
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

if BEST_PP_PARAMS.get("mode") == "dixon_bivar_expected_awmae":
    stage1_team, stage1_opp = dixon_bivariate_select(
        reg_test_team_raw,
        reg_test_opp_raw,
        poisson_test_team_lambda,
        poisson_test_opp_lambda,
        test_outcome_proba,
        BEST_PP_PARAMS,
        full_train_model,
    )
    active_mode = "dixon_bivar_expected_awmae"
else:
    stage1_team, stage1_opp = round_team, round_opp
    active_mode = "round_clip"

stage1_submission = sample[["Id"]].copy()
stage1_submission["team_goals"] = stage1_team
stage1_submission["opp_goals"] = stage1_opp
stage1_submission.to_csv(STAGE1_PATH, index=False)


def make_symmetric_pseudo_test(test_df, team_scores, opp_scores):
    pseudo = test_df.copy()
    pseudo["_pred_team"] = np.asarray(team_scores, dtype=int)
    pseudo["_pred_opp"] = np.asarray(opp_scores, dtype=int)
    pseudo["team_goals"] = pseudo["_pred_team"]
    pseudo["opp_goals"] = pseudo["_pred_opp"]

    for _, grp in pseudo.groupby("match_id", sort=False):
        if len(grp) != 2:
            continue
        idx1, idx2 = grp.index.tolist()
        a = int(round((pseudo.loc[idx1, "_pred_team"] + pseudo.loc[idx2, "_pred_opp"]) / 2))
        b = int(round((pseudo.loc[idx1, "_pred_opp"] + pseudo.loc[idx2, "_pred_team"]) / 2))
        a = int(np.clip(a, 0, BEST_MAX_SCORE))
        b = int(np.clip(b, 0, BEST_MAX_SCORE))
        pseudo.loc[idx1, "team_goals"] = a
        pseudo.loc[idx1, "opp_goals"] = b
        pseudo.loc[idx2, "team_goals"] = b
        pseudo.loc[idx2, "opp_goals"] = a

    pseudo = pseudo.drop(columns=["_pred_team", "_pred_opp"])
    return pseudo


print("Building pseudo-sequential test features...")
test_pseudo = make_symmetric_pseudo_test(test_raw, stage1_team, stage1_opp)
test_pseudo["split_name"] = "test_pseudo"
test_pseudo["can_update_outcome"] = True

pseudo_input = pd.concat([full_train_for_final, test_pseudo], ignore_index=True, sort=False)
pseudo_recon_all = build_reconstructed_features(pseudo_input)
test_pseudo_recon = pseudo_recon_all[pseudo_recon_all["split_name"] == "test_pseudo"].copy()

_, test_pseudo_model, PSEUDO_FEATURE_COLS, PSEUDO_CAT_INDICES = finalize_feature_frames(full_train_recon, test_pseudo_recon, fit_name="pseudo")
assert PSEUDO_FEATURE_COLS == FEATURE_COLS
assert PSEUDO_CAT_INDICES == CAT_FEATURE_INDICES
X_test_pseudo = test_pseudo_model[FEATURE_COLS]

reg_pseudo_team_raw, reg_pseudo_opp_raw = average_regression_predictions(final_models, X_test_pseudo)
poisson_pseudo_team_lambda, poisson_pseudo_opp_lambda = average_poisson_predictions(final_poisson_models, X_test_pseudo)
outcome_pseudo_proba = average_classifier_predictions(final_classifiers, X_test_pseudo)

if BEST_PP_PARAMS.get("mode") == "dixon_bivar_expected_awmae":
    pseudo_team, pseudo_opp = dixon_bivariate_select(
        reg_pseudo_team_raw,
        reg_pseudo_opp_raw,
        poisson_pseudo_team_lambda,
        poisson_pseudo_opp_lambda,
        outcome_pseudo_proba,
        BEST_PP_PARAMS,
        full_train_model,
    )
else:
    pseudo_team, pseudo_opp = postprocess_round_clip(reg_pseudo_team_raw, reg_pseudo_opp_raw, max_score=BEST_MAX_SCORE)

pseudo_submission = sample[["Id"]].copy()
pseudo_submission["team_goals"] = pseudo_team
pseudo_submission["opp_goals"] = pseudo_opp
assert pseudo_submission.shape == sample.shape
assert pseudo_submission["Id"].equals(sample["Id"])
pseudo_submission.to_csv(PSEUDO_PATH, index=False)

# Main output is pseudo stage2.
submission = pseudo_submission.copy()
submission.to_csv(SUBMISSION_PATH, index=False)

changed_stage1_vs_round = ((stage1_submission["team_goals"] != submission_round["team_goals"]) | (stage1_submission["opp_goals"] != submission_round["opp_goals"])).sum()
changed_pseudo_vs_stage1 = ((pseudo_submission["team_goals"] != stage1_submission["team_goals"]) | (pseudo_submission["opp_goals"] != stage1_submission["opp_goals"])).sum()

print("Saved main pseudo submission:", SUBMISSION_PATH)
print("Saved stage1 backup:", STAGE1_PATH)
print("Saved pseudo explicit:", PSEUDO_PATH)
print("Saved round+clip backup:", SUBMISSION_ROUNDCLIP_PATH)
print("Active mode:", active_mode)
print("Validation best Dixon-Coles AW-MAE:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Changed stage1 vs roundclip:", changed_stage1_vs_round)
print("Changed pseudo vs stage1:", changed_pseudo_vs_stage1)
print("Shape:", submission.shape)

display(submission.head())
display(submission[["team_goals", "opp_goals"]].describe())

print("\ndistribusi skor")
print("\nteam_goals")
print(submission["team_goals"].value_counts())
print("\nopp_goals")
print(submission["opp_goals"].value_counts())

print("\ndistribusi pasangan skor")
display(submission[["team_goals", "opp_goals"]].value_counts().head(30).to_frame("count"))


Final regression training recon_cat_mae_d7_seed42 iterations: 1739


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1661994	total: 20.6ms	remaining: 35.8s
150:	learn: 1.0657902	total: 2.78s	remaining: 29.2s
300:	learn: 1.0346089	total: 5.55s	remaining: 26.5s
450:	learn: 1.0206316	total: 8.36s	remaining: 23.9s
600:	learn: 1.0121012	total: 11.2s	remaining: 21.1s
750:	learn: 1.0061714	total: 14s	remaining: 18.4s
900:	learn: 1.0005557	total: 16.8s	remaining: 15.6s
1050:	learn: 0.9953752	total: 19.6s	remaining: 12.8s
1200:	learn: 0.9907992	total: 22.4s	remaining: 10.1s
1350:	learn: 0.9869911	total: 25.3s	remaining: 7.27s
1500:	learn: 0.9830708	total: 28.2s	remaining: 4.47s
1650:	learn: 0.9792977	total: 31.1s	remaining: 1.66s
1738:	learn: 0.9774627	total: 32.8s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1662429	total: 24.8ms	remaining: 43.1s
150:	learn: 1.0658665	total: 2.83s	remaining: 29.8s
300:	learn: 1.0347086	total: 5.55s	remaining: 26.5s
450:	learn: 1.0210968	total: 8.32s	remaining: 23.8s
600:	learn: 1.0122467	total: 11.1s	remaining: 21.1s
750:	learn: 1.0064112	total: 13.9s	remaining: 18.3s
900:	learn: 1.0003987	total: 16.7s	remaining: 15.6s
1050:	learn: 0.9952989	total: 19.5s	remaining: 12.8s
1200:	learn: 0.9908904	total: 22.4s	remaining: 10s
1350:	learn: 0.9872928	total: 25.3s	remaining: 7.26s
1500:	learn: 0.9834156	total: 28.1s	remaining: 4.46s
1650:	learn: 0.9798720	total: 31s	remaining: 1.65s
1738:	learn: 0.9778793	total: 32.7s	remaining: 0us

Final regression training recon_cat_mae_d6_seed7 iterations: 1923


...

0:	learn: 1.1661217	total: 19.8ms	remaining: 30.4s
150:	learn: 1.0700301	total: 3.34s	remaining: 30.6s
300:	learn: 1.0353581	total: 6.64s	remaining: 27.2s
450:	learn: 1.0196145	total: 9.94s	remaining: 23.8s
600:	learn: 1.0101054	total: 13.3s	remaining: 20.6s
750:	learn: 1.0032033	total: 16.6s	remaining: 17.3s
900:	learn: 0.9977478	total: 20s	remaining: 14s
1050:	learn: 0.9919984	total: 23.4s	remaining: 10.7s
1200:	learn: 0.9866301	total: 26.8s	remaining: 7.41s
1350:	learn: 0.9819319	total: 30.2s	remaining: 4.07s
1500:	learn: 0.9780453	total: 33.6s	remaining: 716ms
1532:	learn: 0.9773328	total: 34.3s	remaining: 0us

Final regression training recon_cat_rmse_d7_seed123 iterations: 1089
0:	learn: 1.7756212	total: 17.3ms	remaining: 18.8s
150:	learn: 1.4389151	total: 2.83s	remaining: 17.6s
300:	learn: 1.4024274	total: 5.68s	remaining: 14.9s
450:	learn: 1.3775230	total: 8.54s	remaining: 12.1s
600:	learn: 1.3575928	total: 11.4s	remaining: 9.29s
750:	learn: 1.3416695	total: 14.3s	remaining: 6.4

,Id,team_goals,opp_goals
0,M034984_Seychelles,2,1
1,M034984_Mauritius,1,2
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,1,0


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.079322,1.091698
std,0.848708,0.855172
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,4.000000,4.000000



distribusi skor

team_goals
team_goals
1    18195
0    11495
2    10710
3     1916
4      106
Name: count, dtype: int64

opp_goals
opp_goals
1    18058
0    11369
2    10843
3     2040
4      112
Name: count, dtype: int64

distribusi pasangan skor


count
team_goals opp_goals       
1          1           6965
2          1           6193
1          2           6188
           0           4945
0          1           4811
           2           4636
2          0           4501
0          3           1936
3          0           1818
0          4            112
4          0            105
1          3             97
3          1             88
           2             10
2          2              9
           3              7
4          1              1